# TD MAUP - M2 IGAST

## Comprendre le MAUP sur un cas pratique

Nous allons illustrer le MAUP avec la consommation d'espaces en France métropolitaine, à cinq maillages.

Les données viennent du Cerema. Il s'agit des surfaces d'espaces naturels, agricoles et forestiers (ENAF) consommées entre le 1er janvier 2011 et le 1er janvier 2025, par commune, en hectares.

L'indicateur du TD est la **part du territoire consommée en espaces naturels, agricoles et forestiers entre 2011 et 2025 (%)**. Dans la suite, on l'appelle plus simplement *taux de consommation d'espaces* (colonne `txartif`).

En clair : on compte les hectares de nature, de terres agricoles et de forêts qui ont été transformés en logements, routes, zones d'activité, etc. entre 2011 et 2025, et on les rapporte à la surface totale du territoire. Par exemple, une commune de 1 000 ha qui a perdu 15 ha d'espaces naturels, agricoles ou forestiers au profit de constructions a un taux de 1,5 %.

C'est un flux sur 14 ans, pas la part des sols artificialisés à une date. Une commune qui a beaucoup consommé n'est donc pas forcément « plus artificialisée » qu'une autre. Le [Cerema](https://doc-datafoncier.cerema.fr/doc/guide/conso-enaf/definition-de-la-consommation-despace) détaille la différence entre consommation d'espaces et artificialisation.

Chaque réponse s'appuie sur au moins un chiffre tiré de vos résultats, puis en donne une interprétation et une limite.

## Comment se déroule le TD

Le but est d'**interpréter des résultats**, pas d'apprendre Python. Chaque partie suit le même schéma :

1. **Exemple** : un code complet, écrit pour les communes. Il suffit de l'exécuter et de lire ce qu'il produit. Les commentaires (lignes qui commencent par `#`) expliquent chaque étape.
2. **À vous** : vous refaites la même chose pour les autres mailles. Il suffit de copier l'exemple et de changer le nom de la maille. Vous n'avez rien à programmer.
3. **Questions** : vous interprétez ce que vous voyez.

Quelques repères pour lire le code :

| Ce que vous voyez | Ce que cela veut dire |
|:--|:--|
| `epci_artif = charger('EPCI')` | Le résultat de droite est rangé dans une variable nommée `epci_artif` |
| `gdf['txartif']` | La colonne `txartif` du tableau `gdf` |
| `carte(epci_artif, "EPCI")` | On appelle une fonction déjà écrite, avec ses arguments entre parenthèses |
| `# texte` | Un commentaire, ignoré par Python |

Les cellules s'exécutent dans l'ordre, de haut en bas (`Maj + Entrée`). Une erreur `NameError` signifie en général qu'une cellule précédente n'a pas été exécutée.

## Avant la séance

Le notebook se lit sur GitHub, mais il ne s'y exécute pas. Il s'ouvre dans JupyterLab, dans un environnement Python local. Les consignes pour installer cet environnement avec conda sont dans le [README du dépôt](https://github.com/fbxyz/M2_IGAST/blob/master/README.md#installation-et-utilisation-de-jupyterlab). Les fichiers de données sont aussi dans le dossier `data` du dépôt, à télécharger si la connexion est lente.

---

# 1 - Chargement et contrôle des données

Les cinq maillages regroupent les mêmes 34 735 communes.

| Maille | Fichier | Identifiant | Surface de la maille (ha) |
|:--|:--|:--|:--|
| Communes | `MAUP_COM_ARTIF_3857.geojson` | `insee_com` | `surface_com` |
| EPCI | `MAUP_EPCI_ARTIF_3857.geojson` | `insee_epci` | `surface_epci` |
| Départements | `MAUP_DEP_ARTIF_3857.geojson` | `insee_dep` | `surface_dept` |
| Anciennes régions (avant 2016) | `MAUP_OLDREG_ARTIF_3857.geojson` | `insee_old_reg` | `surface_oldreg` |
| Nouvelles régions (depuis 2016) | `MAUP_REG_ARTIF_3857.geojson` | `insee_reg` | `surface_reg` |

Tous les fichiers contiennent aussi `surface_artif` (surface d'ENAF consommée entre 2011 et 2025, en hectares), `pop11` et `pop22` (population en 2011 et en 2022, Insee).

Le nom de la colonne de surface change d'une maille à l'autre. Le taux se calcule avec toute la précision disponible, `round()` ne sert qu'à l'affichage.

## 1.1 - Communes : *com_artif*

**Exemple à exécuter.** La cellule ci-dessous lit le fichier des communes, change sa projection, puis calcule le taux de consommation d'espaces.

- `gpd.read_file(...)` lit le fichier GeoJSON et crée un *GeoDataFrame*, c'est-à-dire un tableau où chaque ligne est une commune avec sa géométrie.
- `.to_crs(epsg=2154)` change la **projection**. Les fichiers sont en Web Mercator (EPSG:3857), la projection des cartes en ligne, qui étire les territoires vers le nord. On passe en Lambert-93 (EPSG:2154), la projection officielle pour la France métropolitaine, pour que la carte ait la bonne forme. Les surfaces en hectares sont déjà dans les fichiers : la reprojection change l'allure de la carte, pas le taux.
- Le taux est calculé colonne par colonne : surface consommée divisée par surface de la commune, multipliée par 100.

La fonction `charger()` regroupe ces étapes pour n'importe quelle maille. Vous n'avez pas à la modifier, seulement à l'appeler.

In [ ]:
# geopandas lit les fichiers géographiques et les range dans des tableaux (GeoDataFrame)
# https://geopandas.org/
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mapclassify as mc
import numpy as np

sns.set_theme(style="whitegrid")

# Adresse des fichiers : {} sera remplacé par le nom de la maille (COM, EPCI, DEP...)
URL = "https://github.com/fbxyz/M2_IGAST/raw/master/data/MAUP_{}_ARTIF_3857.geojson"
# Texte de légende utilisé dans les cartes et les graphiques
LEG = "Part du territoire consommée 2011-2025 (%)"

# 1. Charger le fichier GeoJSON des communes
com_artif = gpd.read_file(URL.format("COM"))
print(com_artif.crs)  # projection d'origine : EPSG:3857 (Web Mercator)

# 2. Changer la projection vers Lambert-93 (EPSG:2154)
com_artif = com_artif.to_crs(epsg=2154)

# 3. Calculer le taux de consommation d'espaces 2011-2025, en pleine précision (pas de round ici)
com_artif['txartif'] = 100 * com_artif['surface_artif'] / com_artif['surface_com']

# 4. Afficher les premières lignes pour vérifier le résultat
print(com_artif.head())

# Pour chaque maille : nom -> (code du fichier, colonne de surface)
MAILLES = {'Communes': ('COM', 'surface_com'), 'EPCI': ('EPCI', 'surface_epci'),
           'Départements': ('DEP', 'surface_dept'), 'Anciennes régions': ('OLDREG', 'surface_oldreg'),
           'Nouvelles régions': ('REG', 'surface_reg')}


def charger(nom):
    """Lit une maille, la reprojette en Lambert-93 et calcule le taux à partir des surfaces sommées."""
    # Même chose que les étapes 1 à 3 ci-dessus, pour n'importe quelle maille : charger('EPCI')
    fichier, surface = MAILLES[nom]
    gdf = gpd.read_file(URL.format(fichier)).to_crs(epsg=2154)
    gdf['txartif'] = 100 * gdf['surface_artif'] / gdf[surface]
    return gdf

## 1.2 - Les autres mailles

**À vous.** Reprenez la ligne d'exemple des EPCI pour charger les trois autres mailles. Seul le nom entre parenthèses change. Les noms valides sont `'Communes'`, `'EPCI'`, `'Départements'`, `'Anciennes régions'` et `'Nouvelles régions'`, à écrire exactement ainsi (majuscules et accents compris).

> `📝` Chargez les départements, les anciennes et les nouvelles régions dans *dep_artif*, *oldreg_artif* et *reg_artif*, avec le taux dans une colonne *txartif*

In [ ]:
# Exemple : les EPCI
epci_artif = charger('EPCI')

# À vous : retirez le # devant chaque ligne et complétez le nom de la maille
# dep_artif = charger('...')
# oldreg_artif = charger('...')
# reg_artif = charger('...')

## 1.3 - Contrôles

> `📝` Exécutez ces contrôles. Les surfaces sont-elles toutes positives ? Manque-t-il des valeurs ? Les sommes sont-elles identiques d'une maille à l'autre ?

In [ ]:
mailles = {'Communes': com_artif, 'EPCI': epci_artif, 'Départements': dep_artif,
           'Anciennes régions': oldreg_artif, 'Nouvelles régions': reg_artif}

controles = []
for nom, gdf in mailles.items():
    surface = MAILLES[nom][1]
    controles.append({'Maille': nom, 'Unités': len(gdf),
                      'Surfaces nulles ou négatives': int((gdf[surface] <= 0).sum()),
                      'Valeurs manquantes': int(gdf[[surface, 'surface_artif', 'pop11', 'pop22']].isna().sum().sum()),
                      'Surface consommée (ha)': round(gdf['surface_artif'].sum()),
                      'Surface totale (ha)': round(gdf[surface].sum()),
                      'Population 2022': int(gdf['pop22'].sum())})
pd.DataFrame(controles)

---

# 2 - Cartographie des données

Les cinq cartes utilisent les mêmes bornes, la même palette, la même emprise et la même légende. Seul le maillage change d'une carte à l'autre.

Les bornes (0,25, 0,5, 1 et 2 %, puis le maximum communal de 35,7 %) sont des valeurs arrondies qui doublent d'une classe à l'autre, de part et d'autre du taux de la France métropolitaine (0,53 %). La dernière classe couvre toutes les valeurs, quelle que soit la maille.

> `📝` Consultez l'aide de mapclassify si besoin : <https://pysal.org/mapclassify/>

## 2.1 - Carte par communes

**Exemple à exécuter.** La fonction `carte()` dessine la carte d'une maille. Elle applique toujours les mêmes bornes, la même palette et la même légende, ce qui rend les cartes comparables. Il suffit de lui donner le tableau à cartographier, un nom pour le titre et, si on veut, l'épaisseur des contours (plus fins pour les petites unités).

In [ ]:
# Bornes communes à toutes les mailles (limite supérieure incluse) :
# classes de 0 à 0,25 %, de 0,25 à 0,5 %, de 0,5 à 1 %, de 1 à 2 % et de 2 à 36 %
BORNES = [0.25, 0.5, 1, 2, 36]
# Contrôle : aucune valeur ne doit dépasser la dernière borne
assert all(g['txartif'].max() <= BORNES[-1] for g in mailles.values()), "une valeur dépasse la dernière borne"


def carte(gdf, nom, epaisseur=0.1):
    """Carte choroplèthe d'une maille, avec les bornes communes."""
    # ÉTAPE 1 : une figure, mêmes dimensions pour toutes les mailles
    fig, ax = plt.subplots(1, 1, figsize=(9, 6))
    # ÉTAPE 2 : la carte, avec des classes définies par l'utilisateur
    gdf.plot(column='txartif', cmap='OrRd', scheme='user_defined', classification_kwds={'bins': BORNES, 'lowest': 0},
             legend=True, ax=ax, edgecolor='grey', linewidth=epaisseur,
             legend_kwds={'title': LEG, 'loc': 'upper left', 'bbox_to_anchor': (1, 1), 'frameon': True, 'fmt': '{:.2f}'})
    # ÉTAPE 3 : titre, source, sans axes
    ax.set_title(f"Consommation d'espaces 2011-2025, {nom}", fontsize=14, fontweight='bold')
    ax.axis('off')
    ax.text(0.02, 0.02, 'Source : Cerema, 2025', transform=ax.transAxes, fontsize=8, style='italic')
    plt.tight_layout()
    plt.show()


# Exemple : la carte des communes
carte(com_artif, "communes")

## 2.2 - Les autres mailles

**À vous.** Appelez `carte()` pour chaque maille, comme dans l'exemple des EPCI. Plus les unités sont grandes, plus on peut épaissir les contours.

> `📝` Cartographiez les EPCI, les départements, les anciennes et les nouvelles régions avec les mêmes bornes

In [ ]:
# Exemple : les EPCI
carte(epci_artif, "EPCI", epaisseur=0.3)

# À vous : mêmes appels pour dep_artif, oldreg_artif et reg_artif (épaisseurs conseillées : 0.5, 0.8 et 1)

## 2.3 - Questions

> `📝` **Question 1 :** Comment la carte change-t-elle de la commune à la région ?
>
> `📝` **Question 2 :** Les zones qui ont le plus consommé restent-elles les mêmes à chaque maille ?
>
> `📝` **Question 3 :** Quelle maille vous semble la plus pertinente pour étudier la consommation d'espaces ? Pourquoi ?

**Vos réponses ici :**

1.
2.
3.

---

# 3 - Analyse univariée

Décrivez la distribution du taux à chaque maille, avec ses valeurs centrales et sa dispersion.

## 3.1 - Tableau comparatif

**Exemple à exécuter.** La fonction `resume()` calcule les statistiques du taux pour une maille : moyenne, médiane (valeur qui coupe les unités en deux moitiés), écart-type (dispersion autour de la moyenne), minimum et maximum.

**À vous.** Appliquez-la à chaque maille et rassemblez les résultats dans un tableau.

> `📝` Calculez pour chaque maille la moyenne, la médiane, l'écart-type, le minimum et le maximum du taux, puis rassemblez-les dans un tableau

In [ ]:
def resume(nom, gdf):
    """Statistiques descriptives du taux pour une maille."""
    t = gdf['txartif']  # la colonne du taux
    return {'Maille': nom, 'Unités': len(gdf), 'Moyenne': t.mean(), 'Médiane': t.median(),
            'Écart-type': t.std(), 'Minimum': t.min(), 'Maximum': t.max()}


# Exemple : les statistiques des communes
resume('Communes', com_artif)

In [ ]:
# Exemple : le tableau pour deux mailles. pd.DataFrame([...]) transforme une liste de résultats en tableau.
tableau = pd.DataFrame([resume('Communes', com_artif), resume('EPCI', epci_artif)])
tableau.round(2)  # round(2) arrondit à deux décimales pour l'affichage

# À vous : ajoutez dans la liste les trois autres mailles (dep_artif, oldreg_artif, reg_artif)

## 3.2 - Boxplots côte à côte

Rien à écrire ici : exécutez la cellule et lisez le graphique.

Comment lire un boxplot : la boîte contient la moitié centrale des unités, le trait au milieu est la médiane, les moustaches s'étendent vers les valeurs plus rares et les points au-delà sont des valeurs extrêmes. Plus la boîte est haute, plus les unités se ressemblent peu.

Le graphique coupe l'axe à 5 % pour que les boîtes restent lisibles. Le nombre d'unités au-delà est écrit au-dessus de chaque boîte.

In [ ]:
# Un DataFrame au format « long » pour seaborn : une ligne par unité, sa maille et son taux
df_long = pd.concat({nom: gdf['txartif'] for nom, gdf in mailles.items()}, names=['Maille', 'index']).reset_index(name=LEG)

PLAFOND = 5
fig, ax = plt.subplots(1, 1, figsize=(12, 7))
sns.boxplot(data=df_long, x='Maille', y=LEG, ax=ax, hue='Maille', palette='Set2', legend=False)
ax.set_ylim(0, PLAFOND)
for i, (nom, gdf) in enumerate(mailles.items()):
    n = int((gdf['txartif'] > PLAFOND).sum())
    ax.text(i, PLAFOND * 0.97, f"{n} unité(s) au-delà de {PLAFOND} %", ha='center', va='top', fontsize=9, color='firebrick')
ax.set_title("Consommation d'espaces 2011-2025 selon le maillage", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3.3 - Questions

> `📝` **Question 1 :** Comment évoluent la moyenne et la médiane selon la maille ?
>
> `📝` **Question 2 :** Comment évolue la dispersion selon la maille ? Pourquoi ?
>
> `📝` **Question 3 :** Quel lien faites-vous entre ces observations et le MAUP ?

**Vos réponses ici :**

1.
2.
3.

---

# 4 - Analyse bivariée

La consommation d'espaces suit-elle la croissance de la population ? Les fichiers contiennent la population de 2011 et de 2022 (`pop11`, `pop22`). Vous allez mesurer leur corrélation R à chaque maille.

Les périodes diffèrent. La croissance porte sur 2011-2022, la consommation sur 2011-2025.

> `📝` Calculez la croissance de la population entre 2011 et 2022 (en %) pour chaque maille
>
> `📝` Calculez le coefficient de corrélation R entre cette croissance et le taux de consommation d'espaces

## 4.1 - Communes

**Exemple à exécuter.** La fonction `nuage()` fait trois choses pour une maille : elle calcule la croissance de la population entre 2011 et 2022, puis le coefficient de corrélation R avec le taux de consommation d'espaces, puis trace le nuage de points (une unité = un point). R varie de -1 à 1 : proche de 0, les deux variables ne varient pas ensemble de façon linéaire. Il est calculé sur toutes les unités, même si le cadre du graphique en coupe quelques-unes (leur nombre est indiqué sur la figure).

In [ ]:
def nuage(gdf, nom):
    """Calcule R entre croissance de la population et taux de consommation, puis trace le nuage de points."""
    # Quelques unités n'avaient aucun habitant en 2011 : leur croissance n'est pas définie, on les écarte
    d = gdf[gdf['pop11'] > 0].copy()
    # Croissance de la population entre 2011 et 2022, en %
    d['croissance'] = 100 * (d['pop22'] - d['pop11']) / d['pop11']

    # Coefficient de corrélation R entre le taux de consommation et la croissance
    r = d['txartif'].corr(d['croissance'])
    print(f"{nom} : R = {r:.2f} ({len(d)} unités)")

    # Cadre du graphique : les points hors cadre ne sont pas dessinés, mais restent comptés dans R
    XLIM, YLIM = (-40, 80), (0, 5)
    hors = ~(d['croissance'].between(*XLIM) & d['txartif'].between(*YLIM))

    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    # Points plus gros quand il y a peu d'unités
    ax.scatter(d['croissance'], d['txartif'], s=4 if len(d) > 1000 else 40, alpha=0.4, color='firebrick')
    ax.set_title(f"{nom}, R = {r:.2f}", fontsize=14)
    ax.set_xlabel('Croissance de la population 2011-2022 (%)')
    ax.set_ylabel(LEG)
    ax.set_xlim(*XLIM)
    ax.set_ylim(*YLIM)
    ax.text(0.98, 0.97, f"{hors.sum()} unité(s) hors du cadre, comptée(s) dans R", transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='firebrick')
    plt.tight_layout()
    plt.show()


# Exemple : les communes
nuage(com_artif, "Communes")

## 4.2 - Les autres mailles

**À vous.** Appelez `nuage()` pour chaque maille. Le titre du graphique et le texte affiché sous la cellule donnent la valeur de R : notez-la pour chaque maille.

> `📝` Calculez R pour les EPCI, les départements, les anciennes et les nouvelles régions
>
> `📝` Comparez les nuages de points d'une maille à l'autre

In [ ]:
# Exemple : les EPCI
nuage(epci_artif, "EPCI")

# À vous : mêmes appels pour dep_artif, oldreg_artif et reg_artif

## 4.3 - Questions

> `📝` **Question 1 :** Comment évolue R de la commune à la région ?
>
> `📝` **Question 2 :** Que pèse un point quand il ne reste que 13 régions ? Un R proche de zéro montre-t-il l'absence de relation ?
>
> `📝` **Question 3 :** Ce calcul permet-il de dire que la croissance démographique cause la consommation d'espaces ?

**Vos réponses ici :**

1.
2.
3.

---

# Tâche individuelle

Chacun rédige seul une courte réponse (une demi-page) sur un indicateur qui n'a pas été vu en TD, par exemple la part des logements vacants par commune. Quelle représentation choisir ? Quelles classes proposer ? Que pourrait modifier un passage à une autre maille (EPCI, département), dans la carte comme dans les conclusions ?

---

# Pour aller plus loin

1. **L'autocorrélation spatiale.** Comment évolue-t-elle selon la maille ? (indice de Moran avec `esda` de PySAL)
2. **Le changement de découpage de 2016.** Comparez le rang de chaque ancienne région avec celui de la région qui l'a remplacée. Qu'est-ce qui change en même temps que les limites ?
3. **Le lissage spatial.** Remplacez le zonage par une grille régulière et un rayon de voisinage, puis comparez plusieurs rayons

N'oubliez pas de consulter la présentation du cours pour réviser les notions du MAUP.